# Assignment 1
**Research Question**

Does including a book description and numbers of recomendations cause an increase in the number of votes a book receives?

**Motivation**

I would like to see if the book description or numbers of recomendations are related to the amount of votes the books has. 

**Data**

Variables to collect by webscrapping

- Title
- Author
- Rakings
- Votes
- Recommendations
- Descriptions


In [2]:
# Libraries
import requests 
import numpy as np
import pandas as pd
import pprint
from bs4 import BeautifulSoup 

In [3]:
## Webscrapping 
url = "https://books-guru.com/top-100-books"
response = requests.get(url)
soup = BeautifulSoup(response.text)
soup.title

<title>Top 100 Books of All Time – Books Guru</title>

In [4]:
# <span class="field field--name-title field--type-string field--label-hidden">Atomic Habits</span>
title = soup.find_all('span', {'class':'field field--name-title field--type-string field--label-hidden'})
# <div class="field js-amazon-rating field--name-field-amazon-rating field--type-decimal field--label-hidden field__item"> <span class="star-icon"></span> <span class="star-icon"></span> <span class="star-icon"></span> <span class="star-icon active"></span> <span class="star-icon" style="background: linear-gradient(90deg, rgb(242, 201, 76) 80%, rgb(202, 202, 202) 80%);"></span> 4.8</div>
ratings = soup.find_all('div', {'class': 'field--name-field-amazon-rating'}) 
# <div content="123425" class="field field--name-field-amazon-ratings-votes field--type-integer field--label-hidden field__item">123 425 ratings</div>
Q_rates = soup.find_all('div', {'class': 'field field--name-field-amazon-ratings-votes field--type-integer field--label-hidden field__item'}) 
# <div class="field field--name-field-book-author field--type-entity-reference field--label-hidden field__items"> <a href="/authors/james-clear" hreflang="en">James Clear</a></div>
authors = soup.find_all('div', {'class': 'field field--name-field-book-author field--type-entity-reference field--label-hidden field__items'}) 
# <div class="book-description-holder"><div class="field field--name-field-preview-text field--type-text-long field--label-hidden field__item">This book talks about how our personal beliefs and behaviors shape our decisions on money. It is said that knowing money is not mathematics but is closely knitted to human psychology. Armed with witty stories and stories, the author explains why we make such irrational decisions when it comes to money and how to make better ones. It speaks of savings, of the role that luck plays in the acquisition of wealth, and the importance of taking risks, but more importantly, the critical input of thinking long term in the accumulation of wealth. This guide really does offer something new in the context of perspective on wealth, success, and happiness, so it is a must-read for any person battling to better their financial mindset.</div></div>
descriptions = soup.find_all('div', {'class': 'book-description-holder'})

In [ ]:
description = [
    1 if desc.get_text(strip=True) else 0
    for desc in descriptions
]
#len(description)

In [ ]:
## For the recommendations, there is an issue because the scraped information includes both a numeric count and the names of the recommenders. 
## This chunk takes the numeric value and appends it to the recommender names outside the original structure and have the real number of the recomenders.

books = soup.find_all("div", class_="book-hero")
all_recommendations = []

for book in books:
    rec_section = book.find("div", class_="book-recommendations")
    count = 0  # start at 0 for each book

    if rec_section:
        # count all recommendation links without the others
        for link in rec_section.find_all("a", href=True):
            classes = link.get("class") or []
            if "others" not in classes:
                count += 1

        # add the "others" number if it exists
        others_tag = rec_section.find("a", class_="others")
        if others_tag:
            others_text = others_tag.get_text(strip=True)  
            others_number = int(others_text.split()[0])    
            count += others_number

    all_recommendations.append(count)

# all_recommendations

In [ ]:
## creating the dataframe
rows = []

for title, rating, q_rate, author,all_recommendations, description in zip(title, ratings, Q_rates, authors,all_recommendations,description):
    rows.append({
        'title':   title.text.strip(),
        'rating':  rating.text.strip(),
        'votes':   q_rate.text.strip(),
        'author':  author.text.strip(), 
        'recommendations': all_recommendations, 
        'descriptions' : description
    })

df = pd.DataFrame(rows)
df

In [ ]:
## cleaning the data frame 
df['ranking'] = range(1, len(df) + 1)
df['votes'] = df['votes'].str.replace(' ratings', '').str.replace(' ', '').astype(int)
# df['rating'] = df['rating'].astype(float)
df

In [9]:
from scipy import stats
## normalizing the data by doing a z-score for each variable

df['votes_z']   = stats.zscore(df['votes'])
#df['rating_z']  = stats.zscore(df['rating'])
df['recommendations_z'] = stats.zscore(df['recommendations'])

final_df = df
final_df

,title,rating,votes,author,recommendations,descriptions,ranking,votes_z,rating_z,recommendations_z
0,Atomic Habits,4.8,123425,James Clear,9,1,1,0.238470,1.105361,2.013512
1,Man's Search for Meaning,4.7,85596,Viktor Frankl,14,1,2,-0.194890,0.558153,3.680327
2,Sapiens,4.6,136953,Yuval Noah Harari,10,1,3,0.393443,0.010944,2.346875
3,"Thinking, Fast and Slow",4.6,42429,Daniel Kahneman,13,1,4,-0.689402,0.010944,3.346964
4,Think and Grow Rich,4.6,73740,Napoleon Hill,11,1,5,-0.330710,0.010944,2.680238
...,...,...,...,...,...,...,...,...,...,...
95,Midnight Sun,4.8,82211,Stephenie Meyer,0,0,96,-0.233668,1.105361,-0.986754
96,The Underground Railroad,4.4,67475,Colson Whitehead,1,1,97,-0.402480,-1.083473,-0.653391
97,The Catcher in the Rye,4.4,39916,J.D. Salinger,3,1,98,-0.718190,-1.083473,0.013335
98,Extreme Ownership,4.8,36297,"Jocko Willink, Leif Babin",3,1,99,-0.759648,1.105361,0.013335


**Regression Model**:

$$\text{Votes}_i = \beta_0 + \beta_1 \cdot \text{Ranking}_i + \beta_2 \cdot \text{Recommendations}_i + \beta_3 \cdot \text{Description}_i + \epsilon_i$$

**Where:**

- $\text{Votes}_i$ represents the number of votes for book $i$
- $\text{Ranking}_i$ represents the ranking for the book of all time $i$
- $\text{Recommendations}_i$ represents the number of recommendations for book $i$
- $\text{Description}_i$ its a dummy variable where 1 says if it has a description included and 0 if not $i$
m  views.

